# Schweinfurt 2045 regional analysis (v3, better-dimensioned): pre vs optimized HEMS

Complete Schweinfurt mixed-use region (pylovo version 3, voltage-drop-aware dimensioned grids): 66 LV grids, ~5,111 buildings. Head-to-head against the v100 (ampacity-only) baseline notebook. Compares the pre-electrification reference with the 100% electrification optimized-HEMS case. Space heat uses the TEASER source.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy import text
cwd = Path.cwd().resolve()
repo = next(x for x in (cwd, *cwd.parents) if (x / "GridExpand").is_dir())
gx = repo / "GridExpand"
post = gx / "5.postprocessing"
for x in (gx, post):
    if str(x) not in sys.path:
        sys.path.insert(0, str(x))
from common.database import SurroGridDatabase
from powerflow.comparison_data import powerflow_headline_summary_db
db = SurroGridDatabase()
db.pylovo_version_id = "3"
ags = 9662000
pre = powerflow_headline_summary_db(ags=ags, run_name="baseline_static_status_quo_pre_summary_powerflow", stage="pre")
post_opt = powerflow_headline_summary_db(ags=ags, run_name="baseline_static_post_electrification_post-hems-optimized_summary_powerflow", stage="post")
print(f"Pre rows: {len(pre)}, Post rows: {len(post_opt)}")
with db.engine.connect() as c:
    bldg = c.execute(text("""
        SELECT gr.grid_result_id, gr.plz, gr.kcid, gr.bcid,
               COUNT(*) AS n_buildings,
               COUNT(*) FILTER (WHERE COALESCE(UPPER(TRIM(b.building_use)), UPPER(TRIM(b.building_type))) LIKE 'RESIDENTIAL%') AS n_residential
        FROM pylovo.grid_result gr
        JOIN pylovo.municipal_register m ON m.plz = gr.plz
        JOIN pylovo.buildings_result b ON b.grid_result_id = gr.grid_result_id AND b.version_id = gr.version_id
        WHERE m.ags = :ags AND gr.version_id = '3'
        GROUP BY gr.grid_result_id, gr.plz, gr.kcid, gr.bcid
        ORDER BY gr.grid_result_id
    """), {"ags": ags}).mappings().all()
bldg_df = pd.DataFrame(bldg)
pre = pre.merge(bldg_df, left_on="pylovo_grid_result_id", right_on="grid_result_id", how="left")
post_opt = post_opt.merge(bldg_df, left_on="pylovo_grid_result_id", right_on="grid_result_id", how="left")
print(f"Buildings matched: pre={pre['n_buildings'].notna().sum()}/{len(pre)}, post={post_opt['n_buildings'].notna().sum()}/{len(post_opt)}")

## Regional overview

In [ ]:
overview = pd.DataFrame([{
    "Grids": len(pre),
    "Buildings": int(pre['n_buildings'].sum()),
    "Residential": int(pre['n_residential'].sum()),
    "Residential share [%]": 100 * pre['n_residential'].sum() / pre['n_buildings'].sum(),
    "Buildings per grid (median)": int(pre['n_buildings'].median()),
    "Buildings per grid (min)": int(pre['n_buildings'].min()),
    "Buildings per grid (max)": int(pre['n_buildings'].max()),
}])
display(overview.style.hide(axis="index").format({"Residential share [%]": "{:.1f}"}))
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pre['n_buildings'], bins=20, edgecolor='white', color='steelblue')
ax.set(xlabel='Buildings per grid', ylabel='Number of grids', title='Schweinfurt grid-size distribution')
sns.despine()
fig.tight_layout()
plt.show()

## Headline power-flow comparison (regional averages)

Each value is the mean across all 66 grids. Voltage and loading metrics use the worst asset per grid.

In [ ]:
metrics = {
    "Converged timesteps": "n_converged_timesteps",
    "Transformer loading max [%]": "trafo_loading_max_time_percent",
    "Transformer hours >100%": "trafo_loading_hours_above_100",
    "Cable loading max [%]": "cable_loading_max_asset_percent",
    "Cable hours >100% (p95 asset)": "cable_hours_above_100_p95_asset",
    "Voltage min [pu]": "voltage_min_asset_time_pu",
    "Voltage p05 load bus [pu]": "voltage_p05_load_bus_hour_pu",
    "Voltage hours <0.90 (p95 asset)": "voltage_hours_below_0_90_p95_asset",
}
rows = []
for label, col in metrics.items():
    rows.append({"Metric": label, "Pre (mean)": pre[col].mean(), "Post optimized (mean)": post_opt[col].mean(),
                 "Pre (median)": pre[col].median(), "Post optimized (median)": post_opt[col].median(),
                 "Pre (worst)": pre[col].max() if 'hours' in col or 'loading' in col else pre[col].min(),
                 "Post (worst)": post_opt[col].max() if 'hours' in col or 'loading' in col else post_opt[col].min()})
summary = pd.DataFrame(rows).set_index("Metric")
display(summary.style.format("{:.2f}"))

## Voltage and loading distributions across all grids

Each point is one grid. The dashed line marks the 0.90 pu voltage limit and the 100% loading limit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title, ylim in [
    (axes[0], 'voltage_min_asset_time_pu', 'Minimum voltage [pu]', (0.85, 1.02)),
    (axes[1], 'cable_loading_max_asset_percent', 'Max cable loading [%]', (0, 120)),
    (axes[2], 'trafo_loading_max_time_percent', 'Max transformer loading [%]', (0, 120)),
]:
    data = pd.DataFrame({
        'Pre': pre[col], 'Post optimized': post_opt[col],
        'n_buildings': pre['n_buildings']
    })
    for i, (case, color) in enumerate([('Pre', 'steelblue'), ('Post optimized', 'coral')]):
        ax.scatter(data['n_buildings'] + np.random.uniform(-1.5, 1.5, len(data)), data[case],
                   alpha=0.6, s=25, color=color, label=case, zorder=2)
    ax.set(xlabel='Buildings per grid', ylabel=title, title=title)
    if 'voltage' in col:
        ax.axhline(0.90, color='red', linestyle='--', linewidth=0.8, zorder=1)
    else:
        ax.axhline(100, color='red', linestyle='--', linewidth=0.8, zorder=1)
    ax.legend(frameon=False)
    sns.despine()
fig.tight_layout()
plt.show()

## Histograms: pre vs post

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, title, bins in [
    (axes[0], 'voltage_min_asset_time_pu', 'Minimum voltage [pu]', 20),
    (axes[1], 'cable_loading_max_asset_percent', 'Max cable loading [%]', 20),
    (axes[2], 'trafo_loading_max_time_percent', 'Max transformer loading [%]', 20),
]:
    ax.hist(pre[col], bins=bins, alpha=0.6, color='steelblue', label='Pre', edgecolor='white')
    ax.hist(post_opt[col], bins=bins, alpha=0.6, color='coral', label='Post optimized', edgecolor='white')
    ax.set(title=title, xlabel=title.split(' [')[0], ylabel='Number of grids')
    ax.legend(frameon=False)
    sns.despine()
fig.tight_layout()
plt.show()

## Per-grid ranking: worst grids by voltage

In [ ]:
ranking = pd.DataFrame({
    'grid_result_id': pre['pylovo_grid_result_id'].astype(int),
    'n_buildings': pre['n_buildings'].astype(int),
    'Pre voltage min [pu]': pre['voltage_min_asset_time_pu'],
    'Post voltage min [pu]': post_opt['voltage_min_asset_time_pu'],
    'Pre cable max [%]': pre['cable_loading_max_asset_percent'],
    'Post cable max [%]': post_opt['cable_loading_max_asset_percent'],
    'Pre trafo max [%]': pre['trafo_loading_max_time_percent'],
    'Post trafo max [%]': post_opt['trafo_loading_max_time_percent'],
})
display(ranking.sort_values('Post voltage min [pu]').head(15).style.format({
    'Pre voltage min [pu]': '{:.4f}', 'Post voltage min [pu]': '{:.4f}',
    'Pre cable max [%]': '{:.1f}', 'Post cable max [%]': '{:.1f}',
    'Pre trafo max [%]': '{:.1f}', 'Post trafo max [%]': '{:.1f}',
}).background_gradient(subset=['Post voltage min [pu]'], cmap='RdYlGn'))

## Interpretation guardrails

- 66 LV grids covering the Schweinfurt city area (AGS 9662000, PLZ 97422), pylovo version 100.
- Two grids (candidates 30 and 40) are purely commercial with zero residential buildings; they have no heat demand or EV electrification.
- Post cases use six 168-hour TSAM periods; power-flow summaries run all 8,760 timesteps.
- Space heat uses the TEASER source; switch to `infdb_ro_heat` once the INFDB ro_heat schema is loaded for physically calibrated heat profiles.